# AI Powered (Gesture/Voice) Hybrid Transportation Prototye

In [1]:
import cv2
import mediapipe as mp
import numpy as np
from flask import Flask, render_template_string, request
import pyttsx3
import time
import os
import queue
import json
import sounddevice as sd
from vosk import Model, KaldiRecognizer
import threading

C:\Users\MOHAMED ATHIS\anaconda3\envs\botenv\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


### Simulation Mode

In [2]:
SIMULATE = True

if not SIMULATE:
    import serial
    try:
        bluetooth = serial.Serial('COM5', 9600)
        time.sleep(2)
        print("Bluetooth connected.")
    except Exception as e:
        print("Bluetooth Error:", e)
        exit()

### Voice Control

***Speak function (Vosk model) :***

In [3]:
engine = pyttsx3.init()
speech_queue = queue.Queue()

def speak_worker():
    while True:
        text = speech_queue.get()
        if text is None:
            break
        print("Bot:", text)
        engine.say(text)
        engine.runAndWait()
        speech_queue.task_done()

speech_thread = threading.Thread(target=speak_worker, daemon=True)
speech_thread.start()

def speak(text):
    speech_queue.put(text)

Bot: Mobile retro joystick started. Open your browser and go to http://10.139.74.112:5000


***Listener :***

In [4]:
def listen_command():
    try:
        q = queue.Queue()

        def callback(indata, frames, time, status):
            if status:
                print("Error", status)
            q.put(bytes(indata))

        model_path = "vosk-model-small-en-us-0.15"
        if not os.path.exists(model_path):
            speak("Vosk model not found.")
            return None

        model = Model(model_path)
        recognizer = KaldiRecognizer(model, 16000)

        with sd.RawInputStream(samplerate=16000, blocksize=8000, dtype='int16',
                               channels=1, callback=callback):
            print("Listening (Vosk)...")
            speak("Listening...")
            result_text = ""

            timeout_counter = 0
            while True:
                if not q.empty():
                    data = q.get()
                    if recognizer.AcceptWaveform(data):
                        result = json.loads(recognizer.Result())
                        result_text = result.get("text", "")
                        break
                else:
                    timeout_counter += 1
                    time.sleep(0.1)
                    if timeout_counter > 100:  # ~10 sec timeout
                        speak("You didn’t say anything.")
                        return None

        print("You said:", result_text)
        return result_text.lower()

    except Exception as e:
        print("Error:", e)
        speak("Could not recognize your voice.")
        return None

***Voice Command setup:***

In [5]:
def voice_control():
    speak("Voice control activated. Say forward, stop, left, right, or exit.")
    while True:
        command = listen_command()
        if command:
            if "forward" in command:
                send_command('F')
                speak("Moving forward")
            elif "stop" in command:
                send_command('S')
                speak("Stopping")
            elif "left" in command:
                send_command('L')
                speak("Turning left")
            elif "right" in command:
                send_command('R')
                speak("Turning right")
            elif "exit" in command:
                speak("Exiting voice control.")
                break
            else:
                speak("Unknown command")

### Gesture Control

***Gesture Commands:***

In [6]:
def detect_gesture(landmarks):
    finger_tips = [8, 12, 16, 20]  # Index, Middle, Ring, Pinky
    fingers_up = []
    for tip in finger_tips:
        tip_y = landmarks.landmark[tip].y
        pip_y = landmarks.landmark[tip - 2].y
        fingers_up.append(tip_y < pip_y)

    if fingers_up == [True, True, True, True]:
        return "Forward"
    elif fingers_up == [False, False, False, False]:
        return "Stop"
    elif fingers_up == [True, True, False, False]:
        return "Right"
    elif fingers_up == [True, False, False, False]:
        return "Left"
    elif fingers_up == [True, False, False, True]:
        return "Exit"
    else:
        return "Other"

***Gesture Function:***

In [7]:
def gesture_control():
    speak("Gesture control activated. Use your hand to control.")
    mp_hands = mp.solutions.hands
    hands = mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.7)
    mp_draw = mp.solutions.drawing_utils
    cap = cv2.VideoCapture(0)

    last_gesture = ""
    bot_status = ""

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.flip(frame, 1)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb)

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                mp_draw.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
                gesture = detect_gesture(hand_landmarks)

                if gesture != last_gesture:
                    last_gesture = gesture
                    if gesture == "Forward":
                        send_command('F')
                        bot_status = "Moving Forward"
                    elif gesture == "Stop":
                        send_command('S')
                        bot_status = "Stopping"
                    elif gesture == "Left":
                        send_command('L')
                        bot_status = "Turning Left"
                    elif gesture == "Right":
                        send_command('R')
                        bot_status = "Turning Right"
                    elif gesture == "Exit":
                        speak("Exit gesture detected. Closing gesture control.")
                        cap.release()
                        cv2.destroyAllWindows()
                        return
                    else:
                        bot_status = "Unrecognized"

                # Display Gesture & Bot Status
                cv2.putText(frame, f'Gesture: {gesture}', (10, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                cv2.putText(frame, f'Bot: {bot_status}', (10, 70),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

        cv2.imshow("Gesture Control", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()


In [8]:
def mobile_joystick_control():

    app = Flask(__name__)

    html_page = """
    <!DOCTYPE html>
    <html>
    <head>
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Retro Robot Control</title>
        <style>
            body { background: #0d0d0d; color: white; text-align: center; font-family: Arial; margin: 0; }
            h1 { color: #00ffcc; margin-top: 15px; font-size: 1.5em; }
            .mode-btn {
                padding: 12px 18px; margin: 5px;
                background: #1a1a1a; color: white;
                border: 2px solid #00ffcc; border-radius: 10px;
                font-size: 1em; width: 30%;
            }
            .mode-btn:active { background: #00ffcc; color: black; }
            .dpad {
                display: inline-block; margin-top: 30px;
                user-select: none;
            }
            .row { display: flex; justify-content: center; }
            .btn {
                background: radial-gradient(circle at center, #222 0%, #000 100%);
                border: 3px solid #00ffcc;
                border-radius: 12px;
                width: 80px; height: 80px; margin: 8px;
                font-size: 32px; color: #00ffcc;
                box-shadow: 0 0 12px #00ffcc;
                touch-action: none;
            }
            .btn:active {
                background: #00ffcc; color: black;
                box-shadow: 0 0 20px #00ffcc;
            }
        </style>
    </head>
    <body>
        <h1>TalkMotion</h1>

        <!-- Mode Switching -->
        <div>
            <button class="mode-btn" onclick="fetch('/switch_mode?mode=joystick')">Joystick</button>
            <button class="mode-btn" onclick="fetch('/switch_mode?mode=voice')">Voice</button>
            <button class="mode-btn" onclick="fetch('/switch_mode?mode=gesture')">Gesture</button>
        </div>

        <!-- Retro D-Pad -->
        <div class="dpad">
            <div class="row">
                <button class="btn" ontouchstart="startMove('up')" ontouchend="stopMove()">⬆️</button>
            </div>
            <div class="row">
                <button class="btn" ontouchstart="startMove('left')" ontouchend="stopMove()">⬅️</button>
                <div style="width: 80px;"></div>
                <button class="btn" ontouchstart="startMove('right')" ontouchend="stopMove()">➡️</button>
            </div>
            <div class="row">
                <button class="btn" ontouchstart="startMove('down')" ontouchend="stopMove()">⬇️</button>
            </div>
        </div>

        <script>
            var moveInterval;
            function startMove(dir) {
                fetch('/move?dir=' + dir);
                moveInterval = setInterval(() => {
                    fetch('/move?dir=' + dir);
                }, 200); // Keep sending while held
            }
            function stopMove() {
                clearInterval(moveInterval);
                fetch('/move?dir=stop');
            }
        </script>
    </body>
    </html>
    """

    @app.route('/')
    def index():
        return render_template_string(html_page)

    @app.route('/move')
    def move():
        direction = request.args.get('dir')
        if direction == 'up':
            send_command('F')
        elif direction == 'down':
            send_command('B')
        elif direction == 'left':
            send_command('L')
        elif direction == 'right':
            send_command('R')
        elif direction == 'stop':
            send_command('S')
        return "OK"

    @app.route('/switch_mode')
    def switch_mode():
        mode = request.args.get('mode')
        if mode == 'joystick':
            speak("Joystick mode activated.")
        elif mode == 'voice':
            speak("Switching to voice control.")
            threading.Thread(target=voice_control).start()
        elif mode == 'gesture':
            speak("Switching to gesture control.")
            threading.Thread(target=gesture_control).start()
        return "OK"

    def run_app():
        app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

    threading.Thread(target=run_app).start()
    speak("Mobile joystick started. Open your browser and go to http://10.139.74.112:5000")


### Output Type

In [9]:
def send_command(command):
    if SIMULATE:
        print(f"[SIMULATED] Command sent: {command}")
    else:
        bluetooth.write(command.encode())
        print(f"[BT] Sent: {command}")


## Main

In [10]:
def main():
    print("=== AI Robot Assistant ===")
    print("1. Voice Control")
    print("2. Gesture Control")
    print("3. Mobile Joystick Control")
    mode = input("Enter choice: ")

    if mode == '1':
        voice_control()
    elif mode == '2':
        gesture_control()
    elif mode == '3':
        mobile_joystick_control()
    else:
        print("Invalid choice. Please enter 1, 2, or 3.")

if __name__ == "__main__":
    main()


=== AI Robot Assistant ===
1. Voice Control 🎤
2. Gesture Control ✋
3. Mobile Joystick Control 📱


Enter choice:  3


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://10.139.74.112:5000
Press CTRL+C to quit
10.139.74.152 - - [09/Aug/2025 20:49:39] "GET / HTTP/1.1" 200 -
10.139.74.152 - - [09/Aug/2025 20:49:45] "GET / HTTP/1.1" 200 -
10.139.74.152 - - [09/Aug/2025 20:49:47] "GET /move?dir=up HTTP/1.1" 200 -
10.139.74.152 - - [09/Aug/2025 20:49:47] "GET /move?dir=stop HTTP/1.1" 200 -


[SIMULATED] Command sent: F
[SIMULATED] Command sent: S


...

10.139.74.152 - - [09/Aug/2025 20:50:01] "GET /move?dir=up HTTP/1.1" 200 -


[SIMULATED] Command sent: F
[SIMULATED] Command sent: F


10.139.74.152 - - [09/Aug/2025 20:50:02] "GET /move?dir=up HTTP/1.1" 200 -
10.139.74.152 - - [09/Aug/2025 20:50:02] "GET /move?dir=up HTTP/1.1" 200 -
10.139.74.152 - - [09/Aug/2025 20:50:02] "GET /move?dir=stop HTTP/1.1" 200 -


[SIMULATED] Command sent: F
[SIMULATED] Command sent: S


10.139.74.152 - - [09/Aug/2025 20:50:07] "GET /switch_mode?mode=joystick HTTP/1.1" 200 -
